<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/Task3/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#Task 3
## Algorithm Selection Justification

REGRESSION MODELS — Predict exact ACTUAL_COST (£):

1. Linear Regression (Baseline)
   Selected as the interpretable baseline model. NHS prescription
   costs often scale linearly with quantity and drug type. This
   model establishes a performance floor — all other models must
   outperform it to justify their complexity.

2. Random Forest Regressor
   Selected for its ability to capture complex non-linear
   interactions between 15 features across 18M records. NHS
   prescribing patterns vary significantly by region, GP practice
   and drug category — interactions that linear models miss entirely.

CLASSIFICATION MODELS — Predict HIGH_COST flag (£50 threshold):

3. Logistic Regression (Baseline Classifier)
   Selected as the probabilistic baseline classifier. Outputs a
   probability score (0-1) indicating likelihood of high cost —
   directly usable in NHS budget monitoring workflows. Simple,
   fast and highly interpretable.

4. Decision Tree Classifier
   Selected for its unmatched explainability. Produces human-readable
   rules that NHS managers, GPs and policy makers can understand and
   act on without data science expertise. Critical for NHS trust and
   adoption of ML systems.

WHY NOT OTHERS:
- SVM: Does not scale to 18M rows efficiently in PySpark
- KNN: O(n²) complexity — computationally impossible at this scale  
- K-Means: Clustering only — cannot predict cost values
- Deep Learning: GPU required, overkill for structured tabular data

In [1]:
#Task 3 — ML Model Portfolio

#import and start pyspark
!pip install pyspark -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [2]:
#load data from task 2
#mount google drive and load dataset
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed train/test data saved in Task 2
train_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/train_data.parquet'
)
test_df = spark.read.parquet(
    '/content/drive/MyDrive/TASK_DATASET/test_data.parquet'
)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows: {test_df.count():,}")

# Use a smaller sample for model tuning - due to RAM issue
train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample rows: {train_sample.count():,}")
print(f"Testing sample rows: {test_sample.count():,}")

print("Data loaded successfully!")

Mounted at /content/drive
Training rows: 1,469,696
Testing rows: 367,299
Training sample rows: 43,752
Testing sample rows: 11,125
Data loaded successfully!


In [3]:
# Check feature vector dimensions

first_row = train_sample.select("scaled_features").first()

print("Number of features:")
print(len(first_row["scaled_features"]))

Number of features:
87
